# English–Malay code-switching and sentiment in Ayam Gepuk reviews

Research question: How is English–Malay code-switching used to express positive and negative sentiment in online ayam gepuk reviews?

Run all cells from the repository root or Notebooks. Install pandas, openpyxl, matplotlib and a Jupyter Python kernel first. Inputs: annotated workbook, focused_concordance_review.xlsx, selected_evidence_review.xlsx and integrated_discussion_evidence.csv. The latter contains draft supplementary interpretations. Reviewed workbooks are preserved. Outputs are regenerated under Outputs. Missing/incompatible manual review decisions stop the run; do not automatically accept them.

The notebook combines data verification, exploratory N-grams, concordances and contextual/manual sentiment analysis. It does not train a classifier. Phase 1 historical collection counts remain unresolved. Phase 3 interpretations require team review. Source annotations are read-only.


## 1. Load once and audit retained data

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data/CPC 353 Dataset.xlsx").exists())
SOURCE = ROOT / "Data/CPC 353 Dataset.xlsx"
OUT = ROOT / "Outputs/Tables"
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
df = pd.read_excel(SOURCE, sheet_name="Self-Collected Data", header=3, usecols="A:M", keep_default_na=False)
populated = df.apply(lambda col: col.astype(str).str.strip().ne("")).any(axis=1)
df.insert(0, "source_row", range(5, len(df) + 5))
df = df.loc[populated].copy()

# Comparison keys only: original text, punctuation and emojis remain unchanged.
def normalise_for_comparison(text):
    return " ".join(str(text).lower().split())

def audit_records(frame):
    text = frame["Review Text"].astype(str)
    keys = text.map(normalise_for_comparison)
    nonblank = keys.ne("")
    ids = frame["Review ID"].astype(str).str.strip()
    flags = []
    masks = {
        "Blank review text": ~nonblank,
        "Blank review ID": ids.eq(""),
        "Duplicate review ID": ids.ne("") & ids.duplicated(keep=False),
        "Exact duplicate text": nonblank & text.duplicated(keep=False),
        "Normalised duplicate text": nonblank & keys.duplicated(keep=False),
    }
    for issue, mask in masks.items():
        for idx in frame.index[mask]:
            flags.append({"source_row": frame.loc[idx, "source_row"], "review_id": frame.loc[idx, "Review ID"], "issue": issue, "original_text": text.loc[idx], "comparison_key": keys.loc[idx]})
    summary = {
        "retained_reviews_checked": len(frame),
        "blank_review_texts": int((~nonblank).sum()),
        "blank_review_ids": int(ids.eq("").sum()),
        "duplicate_id_excess_rows": int(ids[ids.ne("")].duplicated().sum()),
        "exact_duplicate_text_excess_rows": int(text[nonblank].duplicated().sum()),
        "normalised_duplicate_text_excess_rows": int(keys[nonblank].duplicated().sum()),
    }
    return pd.DataFrame(summary.items(), columns=["check", "count"]), pd.DataFrame(flags, columns=["source_row", "review_id", "issue", "original_text", "comparison_key"])

summary, flags = audit_records(df)
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "phase1_data_checks.csv", index=False)
flags.to_csv(OUT / "phase1_review_flags.csv", index=False, encoding="utf-8-sig")
restaurants = df.groupby("Restaurant Location", sort=False).size().reset_index(name="retained_reviews")
restaurants.to_csv(OUT / "phase1_restaurant_counts.csv", index=False, encoding="utf-8-sig")
provenance = pd.DataFrame([
    ["Five main collection runs", "275", "Reported in shared report; unverified", "Run manifests/raw exports not found in inspected workspace or Git history"],
    ["Approximate original collection", "around 300", "Reported in shared report; unverified", "May be rounding or a different collection scope; cannot determine from available files"],
    ["Retained annotated reviews", str(len(df)), "Verified from source workbook", "Self-Collected Data sheet; populated rows only"],
    ["Historical duplicates removed", "", "Unknown", "A clean final dataset does not establish how many records were previously removed"],
    ["Historical empty reviews removed", "", "Unknown", "Requires collection/cleaning records"],
    ["Historical other exclusions", "", "Unknown", "Requires collection/cleaning records"],
], columns=["stage", "count_or_claim", "status", "evidence_or_limitation"])
provenance.to_csv(OUT / "phase1_collection_provenance.csv", index=False, encoding="utf-8-sig")
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest() == source_hash
print(summary.to_string(index=False))
print(restaurants.to_string(index=False))
print("Source workbook unchanged:", source_hash)


                                check  count
             retained_reviews_checked    197
                   blank_review_texts      0
                     blank_review_ids      0
             duplicate_id_excess_rows      0
     exact_duplicate_text_excess_rows      0
normalised_duplicate_text_excess_rows      0
          Restaurant Location  retained_reviews
    Ayam Gepuk Macet Chow Kit                48
Ayam Gepuk Artisan Alam Damai                44
          Ayam Gepuk Original                37
        Gepuklah by mingchuun                19
Ayam Gepuk Pak Gembus Bangsar                49
Source workbook unchanged: bd587042d3bd3d1c514c9bf7dfa41bbbe92de58d676b666aa9fb12afb3f54dae


In [2]:
import re
import matplotlib.pyplot as plt
PROJECT_DIR = ROOT
INPUT_FILE = SOURCE
TABLE_DIR = OUT
FIGURE_DIR = ROOT/'Outputs/Figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENTS = ['Positive','Negative','Neutral']
SWITCH_TYPES = ['Intra-sentential','Inter-sentential','Tag Switching','N/A']
COLORS = ['#287D8E','#B65044','#8B8D91']
protected = [SOURCE, TABLE_DIR/'focused_concordance_review.xlsx', TABLE_DIR/'selected_evidence_review.xlsx', TABLE_DIR/'integrated_discussion_evidence.csv']
protected_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in protected}


In [3]:
def validate_reviews(data):
    if data.empty:
        raise ValueError("No reviews available.")
    if data["Review ID"].eq("").any() or data["Review ID"].duplicated().any():
        raise ValueError("Review IDs must be present and unique.")
    if data["Review Text"].eq("").any():
        raise ValueError("Review text is missing.")
    for column, allowed in [("Sentiment", SENTIMENTS),
                            ("Code-Switching Type", SWITCH_TYPES),
                            ("Language Classification", ["English", "Malay", "Code-switched"])]:
        invalid = data.loc[~data[column].isin(allowed), ["Review ID", column]]
        if not invalid.empty:
            raise ValueError(f"Resolve invalid or missing {column} labels:\n{invalid}")
    inconsistent = data["Language Classification"].eq("Code-switched") == data["Code-Switching Type"].eq("N/A")
    if inconsistent.any():
        raise ValueError("Language classification and switching type disagree.")

validate_reviews(df)
print(f"Validated {len(df)} reviews; all sentiment labels are present.")


Validated 197 reviews; all sentiment labels are present.


## 2. Exploratory N-grams
Frequency does not establish bilingual wording. Sentiment breakdowns describe whole-review labels.

# Phase 2: exploratory bigrams and trigrams

Find recurring sequences in the code-switched subset to support contextual analysis of sentiment. A phrase from a code-switched review is not necessarily bilingual. No automatic phrase-language or phrase-sentiment classification is performed.

Rules: lowercase token copies; no stop-word removal, stemming or spelling correction. Retain negation, contrast markers and informal forms such as betul2. Internal apostrophes and hyphens stay in tokens; curly apostrophes are normalised. Decimal numbers remain tokens. Boundaries are full stops (except between digits), question/exclamation marks, common Unicode equivalents, ellipses, semicolons and line breaks. Abbreviation full stops may over-segment. Commas, colons, brackets and quote/dash separators do not break adjacency; other symbols, including emojis, break token runs. Thus extraction stays within punctuation-defined segments and token runs, never crossing reviews. This conservative heuristic is not a linguistic sentence parser.

Count overlapping two/three-token windows. Rank by occurrences, distinct reviews, then alphabetically. Recurring means at least two distinct reviews. Food names remain in frequency counts but alone do not prove switching. Sentiment breakdowns refer to whole-review labels, not phrase polarity. Run from the repository root or a child directory with pandas/openpyxl. Source annotations remain unchanged.


In [4]:
cs=df.loc[df['Language Classification'].eq('Code-switched')].copy()

In [5]:
BOUNDARY = re.compile(r'(?<!\d)\.|\.(?!\d)|[!?…‼⁉。！？;；\r\n]+')
TOKEN = re.compile(r"\d+(?:[.,]\d+)+|[^\W_]+(?:['’\-][^\W_]+)*", re.UNICODE)
GAP = re.compile(r"[\s,:，：()\[\]{}\"'‘’“”\-–—]*")
def token_runs(text):
    start=0
    for sid,boundary in enumerate([*BOUNDARY.finditer(text),None],1):
        stop=boundary.start() if boundary else len(text)
        run=[]; rid=1
        for token in TOKEN.finditer(text,start,stop):
            if run and not GAP.fullmatch(text[run[-1].end():token.start()]):
                yield sid,rid,start,stop,run
                run=[];rid+=1
            run.append(token)
        if run:yield sid,rid,start,stop,run
        start=boundary.end() if boundary else len(text)
def extract_ngrams(text,sizes=(2,3)):
    for sid,rid,start,stop,tokens in token_runs(text):
        for n in sizes:
            for i in range(len(tokens)-n+1):
                w=tokens[i:i+n]
                yield dict(n=n,ngram=' '.join(t.group().lower().replace('’',"'") for t in w),segment_id=sid,run_id=rid,match_start=w[0].start(),match_end=w[-1].end(),source_quote=text[w[0].start():w[-1].end()],segment_text=text[start:stop])
rows=[]
for _,r in cs.iterrows():
    for h in extract_ngrams(r['Review Text']):
        rows.append(dict(review_id=r['Review ID'],source_row=r['source_row'],sentiment=r['Sentiment'],primary_switching_type=r['Code-Switching Type'],**h))
hits=pd.DataFrame(rows)
assert not hits.empty
assert not hits.duplicated(['review_id','n','match_start','match_end']).any()
lookup=cs.set_index('Review ID')['Review Text']
for h in hits.itertuples():assert lookup.loc[h.review_id][h.match_start:h.match_end]==h.source_quote
print(hits.groupby('n').size().to_string())


n
2    7610
3    6707


In [6]:
summary=hits.groupby(['n','ngram']).agg(occurrences=('review_id','size'),distinct_reviews=('review_id','nunique')).reset_index().sort_values(['n','occurrences','distinct_reviews','ngram'],ascending=[True,False,False,True])
summary['review_percentage']=(100*summary['distinct_reviews']/len(cs)).round(2)
summary['review_denominator']=len(cs)
recurring=summary.loc[summary['distinct_reviews'].ge(2)].copy()
top=summary.groupby('n',group_keys=False).head(20)
by_sentiment=hits.groupby(['n','ngram','sentiment']).agg(occurrences=('review_id','size'),distinct_reviews=('review_id','nunique')).reset_index()
by_sentiment['review_denominator']=by_sentiment['sentiment'].map(cs['Sentiment'].value_counts())
by_sentiment['within_sentiment_review_percentage']=(100*by_sentiment['distinct_reviews']/by_sentiment['review_denominator']).round(2)
assert summary['occurrences'].sum()==len(hits)==by_sentiment['occurrences'].sum()
assert summary['distinct_reviews'].le(summary['occurrences']).all()
print(top[['n','ngram','occurrences','distinct_reviews']].to_string(index=False))


 n              ngram  occurrences  distinct_reviews
 2         ayam gepuk           60                48
 2             it was           14                 8
 2           kedai ni           13                10
 2         first time           12                 9
 2          puas hati           11                10
 2            this is           11                 9
 2             of the           11                 8
 2             on the           11                 6
 2       kobis goreng           10                 9
 2       kubis goreng           10                 8
 2         the sambal           10                 7
 2       boleh tambah            9                 9
 2           ayam dia            9                 7
 2            dine in            8                 8
 2         and sambal            8                 7
 2            and the            8                 7
 2      gepuk artisan            8                 7
 2        tambah nasi            8            

In [7]:
def phrases(text):return [h['ngram'] for h in extract_ngrams(text,(2,))]
assert phrases('sedap. service bagus')==['service bagus']
assert phrases('sedap\nservice bagus')==['service bagus']
assert phrases('sedap‼ service bagus')==['service bagus']
assert phrases('tak sedap but not bad')==['tak sedap','sedap but','but not','not bad']
assert phrases('very very very')==['very very','very very']
assert phrases('sedap 😊 service bagus')==['service bagus']
assert phrases('ayam, juicy')==['ayam juicy']
assert phrases("don't regret")==["don't regret"]
assert phrases('harga 9.90 murah')==['harga 9.90','9.90 murah']
assert phrases('check betul2')==['check betul2']
assert phrases('one')==[]
for table,name in [(hits,'ngram_occurrences.csv'),(summary,'ngram_frequencies.csv'),(recurring,'ngram_recurring.csv'),(top,'ngram_top20.csv'),(by_sentiment,'ngram_by_review_sentiment.csv')]:
    table.to_csv(OUT/name,index=False,encoding='utf-8-sig')
assert hashlib.sha256(SOURCE.read_bytes()).hexdigest()==source_hash
print('Boundary, negation, overlap, token and source-offset checks passed; source unchanged.')
print('Recurring phrases:',recurring.groupby('n').size().to_dict())


Boundary, negation, overlap, token and source-offset checks passed; source unchanged.
Recurring phrases: {2: 608, 3: 99}


## 3. Concordance and M1 reviewed evidence
Local switching events and primary review labels are different units.

In [8]:
# Confirmed expected columns exist

required_columns = [
    "Review ID",
    "Restaurant Location",
    "Review Text",
    "Language Classification",
    "Code-Switching Type",
    "Notes / Annotator Comments",
    "Sentiment",
]

missing_columns = set(required_columns) - set(df.columns)

assert not missing_columns, (
    f"Missing expected columns: {sorted(missing_columns)}"
)

## Check data quality

### Identify issues without changing records

In [9]:
issues = []

for column in required_columns:
    blank = (df[column].isna() | df[column].astype("string").str.strip().eq("").fillna(False))

    for _, row in df.loc[blank].iterrows():
        issues.append({
            "source_row": row["source_row"],
            "review_id": row["Review ID"],
            "field": column,
            "issue": "Missing or empty value",
        })

for column in ["Review ID", "Review Text"]:
    duplicated = df[column].notna() & df[column].duplicated(keep=False)

    for _, row in df.loc[duplicated].iterrows():
        issues.append({
            "source_row": row["source_row"],
            "review_id": row["Review ID"],
            "field": column,
            "issue": "Exact duplicate; review before taking action",
        })

issue_log = pd.DataFrame(
    issues,
    columns=["source_row", "review_id", "field", "issue"],
)

issue_log.to_csv(
    TABLE_DIR / "data_review_flags.csv",
    index=False,
    encoding="utf-8-sig",
)

display(issue_log)

# IDs and text are necessary for reliable extraction.
if (
    df["Review ID"].astype("string").str.strip().eq("").any()
    or df["Review ID"].duplicated().any()
):
    raise ValueError("Review missing or duplicate IDs before continuing.")

if df["Review Text"].astype("string").str.strip().eq("").any():
    raise ValueError("Review empty texts before continuing.")

In [10]:
# Display the actual count
print("Total reviews:", len(df), "\n")
print(df["Language Classification"].value_counts(dropna=False), "\n")
print(df["Code-Switching Type"].value_counts(dropna=False))

Total reviews: 197 

Language Classification
Code-switched    144
English           36
Malay             17
Name: count, dtype: int64 

Code-Switching Type
Intra-sentential    139
N/A                  53
Inter-sentential      5
Name: count, dtype: int64


## Calculate language and switching distribution

In [11]:
cs = df.loc[df["Language Classification"].eq("Code-switched")].copy()

assert len(cs) > 0, "No code-switched reviews were found."

# Analysis for language type
language_summary = (
    df["Language Classification"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="review_count")
)

language_summary["percentage"] = (language_summary["review_count"] / len(df) * 100).round(2)

# Analysis for code-switching type
switching_summary = (
    cs["Code-Switching Type"]
    .value_counts()
    .reindex(["Intra-sentential", "Inter-sentential", "Tag Switching"], fill_value=0,)
    .rename_axis("primary_switching_type")
    .reset_index(name="review_count")
)

switching_summary["percentage"] = (switching_summary["review_count"] / len(cs) * 100).round(2)

display(language_summary)
display(switching_summary)

## Define searches and generate KWIC

### Search terms

In [12]:
search_terms = {
    "sedap": ["sedap", "sedapp", "sedappp", "sedapppppp"],
    "pedas": ["pedas"],
    "service": ["service"],
    "tapi": ["tapi"],
    "but": ["but"],
    "repeat": ["repeat"],
    "portion": ["portion"],
    "refill": ["refill"],
    "juicy": ["juicy"],
    "worth": ["worth"],
}

search_register = pd.DataFrame([
    {
        "search_term": term,
        "included_forms": " | ".join(forms),
        "case_sensitive": False,
        "context_tokens_each_side": 5,
    }
    for term, forms in search_terms.items()
])

display(search_register)

### Extraction function

In [13]:
# KWIC function
def extract_kwic(text, pattern, window=5):
    """Return source-preserving context for each regex match."""
    tokens = list(re.finditer(r"\S+", text))

    for match in pattern.finditer(text):
        overlapping = [
            i for i, token in enumerate(tokens)
            if token.start() < match.end()
            and token.end() > match.start()
        ]

        if not overlapping:
            continue

        first_token = overlapping[0]
        last_token = overlapping[-1]

        left_index = max(0, first_token - window)
        right_index = min(len(tokens) - 1, last_token + window)

        context_start = tokens[left_index].start()
        context_end = tokens[right_index].end()

        yield {
            "match_start": match.start(),
            "match_end": match.end(),
            "left_context": text[context_start:match.start()],
            "matched_form": match.group(),
            "right_context": text[match.end():context_end],
        }

### Generate the table

In [14]:
# Result of KWIC
kwic_rows = []

for _, review in cs.iterrows():
    text = review["Review Text"]

    if not isinstance(text, str) or not text.strip():
        continue

    for term, forms in search_terms.items():
        alternatives = "|".join(
            re.escape(form)
            for form in sorted(set(forms), key=len, reverse=True)
        )

        pattern = re.compile(
            rf"(?<!\w)(?:{alternatives})(?!\w)",
            flags=re.IGNORECASE,
        )

        for occurrence in extract_kwic(text, pattern):
            kwic_rows.append({
                "review_id": review["Review ID"],
                "source_row": review["source_row"],
                "search_term": term,
                **occurrence,
                "primary_switching_type":
                    review["Code-Switching Type"],
                "restaurant": review["Restaurant Location"],
                "sentiment": review["Sentiment"],
                "existing_notes":
                    review["Notes / Annotator Comments"],
                "full_review": text,
            })

kwic = pd.DataFrame(kwic_rows)

kwic.insert(
    0,
    "hit_id",
    [
        f"{row.review_id}:{row.search_term}:{row.match_start}"
        for row in kwic.itertuples()
    ],
)

kwic.head(10)

## Verify and summarise keyword results

In [15]:
assert kwic["hit_id"].is_unique

for row in kwic.itertuples():
    assert (
        row.full_review[row.match_start:row.match_end]
        == row.matched_form
    )

keyword_summary = (
    kwic.groupby("search_term")
    .agg(
        occurrences=("hit_id", "size"),
        distinct_reviews=("review_id", "nunique"),
    )
    .reset_index()
    .sort_values("distinct_reviews", ascending=False)
)

display(keyword_summary)

kwic.to_csv(
    TABLE_DIR / "kwic_all_hits.csv",
    index=False,
    encoding="utf-8-sig",
)

keyword_summary.to_csv(
    TABLE_DIR / "keyword_search_summary.csv",
    index=False,
)

search_register.to_csv(
    TABLE_DIR / "search_register.csv",
    index=False,
)

### Extract the focused concordance results

In [16]:
FOCUSED_TERMS = ["repeat", "portion"]

focused_kwic = kwic.loc[
    kwic["search_term"].isin(FOCUSED_TERMS)
].copy()

focused_kwic = focused_kwic.sort_values(
    ["search_term", "review_id", "match_start"]
).reset_index(drop=True)

if focused_kwic.empty:
    raise ValueError("No focused hits found. Run the original KWIC extraction first.")

if not focused_kwic["hit_id"].is_unique:
    raise ValueError("Duplicate hit IDs found in the focused results.")

focused_counts = (
    focused_kwic.groupby("search_term")
    .agg(
        occurrences=("hit_id", "size"),
        distinct_reviews=("review_id", "nunique"),
    )
    .reindex(FOCUSED_TERMS, fill_value=0)
    .reset_index()
)

print("Total focused keyword hits:", len(focused_kwic))
display(focused_counts)

Total focused keyword hits: 34


### Prepare wider context and review suggestions

In [17]:
def context_from_offsets(text, start, end, window=10):
    """Return wider context around a previously located keyword."""
    tokens = list(re.finditer(r"\S+", text))

    overlapping = [
        i
        for i, token in enumerate(tokens)
        if token.start() < end and token.end() > start
    ]

    if not overlapping:
        raise ValueError("Keyword position does not overlap a token.")

    left_index = max(0, overlapping[0] - window)
    right_index = min(
        len(tokens) - 1,
        overlapping[-1] + window,
    )

    context_start = tokens[left_index].start()
    context_end = tokens[right_index].end()

    return pd.Series({
        "wide_left_context": text[context_start:start],
        "wide_right_context": text[end:context_end],
        "wide_context": text[context_start:context_end],
    })


wider_context = focused_kwic.apply(
    lambda row: context_from_offsets(
        row["full_review"],
        int(row["match_start"]),
        int(row["match_end"]),
        window=10,
    ),
    axis=1,
)

context_columns = [
    "wide_left_context",
    "wide_right_context",
    "wide_context",
]

# Remove previous context columns, including duplicates.
focused_kwic = focused_kwic.drop(
    columns=context_columns,
    errors="ignore",
)

# Assign one set of context columns.
focused_kwic[context_columns] = wider_context[context_columns]

display(
    focused_kwic[
        [
            "review_id",
            "search_term",
            "wide_left_context",
            "matched_form",
            "wide_right_context",
        ]
    ].head(20)
)

In [18]:
review_guidance = {
    "repeat": (
        "Check whether the keyword combines with Malay wording "
        "within a sentence, forms a separate English return expression "
        "after a Malay segment, or occurs in English-only local context."
    ),
    "portion": (
        "Check whether the English noun is combined with a Malay "
        "quantity/evaluation phrase, occurs across a language-changing "
        "sentence boundary, or appears in English-only local context."
    ),
}

focused_kwic["review_guidance"] = (
    focused_kwic["search_term"].map(review_guidance)
)

focused_kwic.to_csv(
    TABLE_DIR / "focused_kwic_hits.csv",
    index=False,
    encoding="utf-8-sig",
)

### Create the protected review workbook

In [19]:
FOCUSED_REVIEW_FILE = (
    TABLE_DIR / "focused_concordance_review.xlsx"
)

focused_review_fields = [
    "evidence_status",
    "pattern_group",
    "observed_switch_type",
    "review_note",
]


def verify_focused_workbook(form):
    """Check that review decisions belong to the current extraction."""
    required = {
        "hit_id",
        "full_review",
        *focused_review_fields,
    }

    missing = required - set(form.columns)
    if missing:
        raise ValueError(
            f"Focused review workbook is missing: {sorted(missing)}"
        )

    if form["hit_id"].duplicated().any():
        raise ValueError("Duplicate hit IDs in the review workbook.")

    if set(form["hit_id"]) != set(focused_kwic["hit_id"]):
        raise ValueError(
            "The workbook and current focused hits differ. "
            "Preserve the workbook and reconcile versions before continuing."
        )

    saved_text = form.set_index("hit_id")["full_review"]
    current_text = focused_kwic.set_index("hit_id")["full_review"]

    if not saved_text.reindex(current_text.index).eq(current_text).all():
        raise ValueError(
            "Source text has changed since the review workbook was created."
        )


if FOCUSED_REVIEW_FILE.exists():
    existing_review = pd.read_excel(
        FOCUSED_REVIEW_FILE,
        sheet_name="Review",
        keep_default_na=False,
    )

    verify_focused_workbook(existing_review)

    print("Existing focused review workbook preserved.")

else:
    review_form = focused_kwic.copy()

    for field in focused_review_fields:
        review_form[field] = ""

    # Put the most useful columns first.
    first_columns = [
        "hit_id",
        "review_id",
        "search_term",
        "wide_context",
        *focused_review_fields,
        "review_guidance",
        "full_review",
        "existing_notes",
    ]

    remaining_columns = [
        column
        for column in review_form.columns
        if column not in first_columns
    ]

    review_form = review_form[
        first_columns + remaining_columns
    ]

    with pd.ExcelWriter(
        FOCUSED_REVIEW_FILE,
        engine="openpyxl",
    ) as writer:
        review_form.to_excel(
            writer,
            sheet_name="Review",
            index=False,
        )

        sheet = writer.sheets["Review"]
        sheet.freeze_panes = "D2"
        sheet.auto_filter.ref = sheet.dimensions

    print("Created focused review workbook.")

print("Open this file:")
print(FOCUSED_REVIEW_FILE.resolve())

Existing focused review workbook preserved.
Open this file:
D:\PROJECT\CPC353 Assignment 1\CPC-354-Assignment-1\Outputs\Tables\focused_concordance_review.xlsx


### Import, validate, and summarize decisions

In [20]:
focused_review = pd.read_excel(
    FOCUSED_REVIEW_FILE,
    sheet_name="Review",
    keep_default_na=False,
)

verify_focused_workbook(focused_review)

# Only import your new interpretation fields.
# Original text and labels come from the current KWIC table.
focused_analysis = focused_kwic.merge(
    focused_review[["hit_id", *focused_review_fields]],
    on="hit_id",
    how="left",
    validate="one_to_one",
)

for field in focused_review_fields:
    focused_analysis[field] = (
        focused_analysis[field].astype(str).str.strip()
    )

allowed_statuses = {
    "Confirmed",
    "No local switching",
    "Ambiguous",
}

allowed_confirmed_types = {
    "Intra-sentential",
    "Inter-sentential",
    "Both",
}

review_flags = []

for _, row in focused_analysis.iterrows():
    reasons = []
    status = row["evidence_status"]

    if not status:
        reasons.append("Pending review")
    elif status not in allowed_statuses:
        reasons.append("Unexpected evidence status")

    if status == "Confirmed":
        if not row["pattern_group"]:
            reasons.append("Confirmed hit needs a pattern group")

        if row["observed_switch_type"] not in allowed_confirmed_types:
            reasons.append(
                "Confirmed hit needs a valid observed switching type"
            )

    if status == "Ambiguous" and not row["review_note"]:
        reasons.append("Explain what is ambiguous")

    if reasons:
        review_flags.append({
            "hit_id": row["hit_id"],
            "review_id": row["review_id"],
            "issue": "; ".join(reasons),
        })

focused_flags = pd.DataFrame(
    review_flags,
    columns=["hit_id", "review_id", "issue"],
)

focused_flags.to_csv(
    TABLE_DIR / "focused_review_flags.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    focused_analysis["evidence_status"]
    .replace("", "Pending")
    .value_counts()
)

display(focused_flags)

if not focused_flags.empty:
    raise ValueError(
        "Complete or correct the flagged rows in Excel, save the file, "
        "and rerun this cell."
    )

### Evidence status summary

In [21]:
status_order = [
    "Confirmed",
    "No local switching",
    "Ambiguous",
]

focused_status_summary = pd.crosstab(
    focused_analysis["search_term"],
    focused_analysis["evidence_status"],
).reindex(
    index=FOCUSED_TERMS,
    columns=status_order,
    fill_value=0,
)

focused_status_summary["total_keyword_hits"] = (
    focused_status_summary[status_order].sum(axis=1)
)

focused_status_summary["confirmed_hit_percentage"] = (
    focused_status_summary["Confirmed"]
    / focused_status_summary["total_keyword_hits"].replace(0, float("nan"))
    * 100
).round(2)

focused_status_summary = (
    focused_status_summary
    .reset_index()
    .rename_axis(None, axis=1)
)

display(focused_status_summary)

### Visualise focused concordance results

In [ ]:
from matplotlib.ticker import MaxNLocator
chart_data = focused_status_summary.copy()

assert chart_data["Ambiguous"].sum() == 0, (
    "Resolve ambiguous hits or include them in the chart."
)

terms = chart_data["search_term"].tolist()
confirmed = chart_data["Confirmed"].to_numpy()
no_local_switching = chart_data["No local switching"].to_numpy()
totals = confirmed + no_local_switching

assert (totals == chart_data["total_keyword_hits"].to_numpy()).all(), (
    "Chart counts do not match the total keyword hits."
)

# Create a compact chart
fig, ax = plt.subplots(figsize=(4.2, 2.6), dpi=150)

ax.barh(
    terms,
    confirmed,
    label="Confirmed switching",
    color="#4C78A8",
)

ax.barh(
    terms,
    no_local_switching,
    left=confirmed,
    label="No local switching",
    color="#D9D9D9",
)

# Add counts inside the bars and totals beside them.
for i, (yes, no, total) in enumerate(
    zip(confirmed, no_local_switching, totals)
):
    if yes > 0:
        ax.text(
            yes / 2, i, str(int(yes)),
            ha="center", va="center",
            color="white", fontsize=10,
        )

    if no > 0:
        ax.text(
            yes + no / 2, i, str(int(no)),
            ha="center", va="center",
            color="black", fontsize=10,
        )

    ax.text(
        total + 0.3, i, f"n = {int(total)}",
        ha="left", va="center", fontsize=9,
    )

ax.invert_yaxis()
ax.set_xlabel("Number of keyword hits")
ax.set_xlim(0, max(totals.max(), 1) * 1.25)
ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=5))

ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", length=0)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.25),
    ncol=2,
    frameon=False,
    fontsize=8,
)

fig.tight_layout()

# Save using the output folder defined earlier in the notebook.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
chart_path = FIGURE_DIR / "focused_concordance_results.png"

fig.savefig(
    chart_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(fig)

### Confirmed pattern summary

In [22]:
focused_confirmed = focused_analysis.loc[
    focused_analysis["evidence_status"].eq("Confirmed")
].copy()

focused_pattern_summary = (
    focused_confirmed.groupby(
        ["search_term", "pattern_group"],
        dropna=False,
    )
    .agg(
        confirmed_keyword_occurrences=("hit_id", "nunique"),
        distinct_reviews=("review_id", "nunique"),
        example_review_ids=(
            "review_id",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
    .reset_index()
    .sort_values(
        ["search_term", "distinct_reviews"],
        ascending=[True, False],
    )
)

display(focused_pattern_summary)

### Export summaries and examples for discussion

#### Select a small set of examples automatically

In [23]:
confirmed_examples = (
    focused_confirmed
    .sort_values(["search_term", "pattern_group", "review_id", "match_start"])
    .drop_duplicates(["search_term", "pattern_group", "review_id"])
    .groupby(["search_term", "pattern_group"], group_keys=False)
    .head(2)
)

other_examples = (
    focused_analysis.loc[
        focused_analysis["evidence_status"].isin(
            ["No local switching", "Ambiguous"]
        )
    ]
    .sort_values(["search_term", "evidence_status", "review_id"])
    .groupby(["search_term", "evidence_status"], group_keys=False)
    .head(1)
)

focused_examples = pd.concat(
    [confirmed_examples, other_examples],
    ignore_index=True,
).drop_duplicates("hit_id")

example_columns = [
    "hit_id",
    "review_id",
    "search_term",
    "wide_context",
    "evidence_status",
    "pattern_group",
    "observed_switch_type",
    "review_note",
    "full_review",
]

display(focused_examples[example_columns])

#### Export the results

In [24]:
focused_analysis.to_csv(
    TABLE_DIR / "focused_concordance_analysis.csv",
    index=False,
    encoding="utf-8-sig",
)

focused_status_summary.to_csv(
    TABLE_DIR / "focused_evidence_status_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

focused_pattern_summary.to_csv(
    TABLE_DIR / "focused_pattern_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

focused_examples[example_columns].to_csv(
    TABLE_DIR / "focused_examples.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Focused concordance outputs saved.")

Focused concordance outputs saved.


## Create a small, prefilled review workbook

### Prepare selected evidence

In [25]:
SELECTED_REVIEW_FILE = TABLE_DIR / "selected_evidence_review.xlsx"

if df["Review ID"].duplicated().any():
    raise ValueError("Duplicate review IDs must be reviewed first.")

source = df.set_index("Review ID")

# Example of the review
evidence_specs = [
    (
        "E01", "REV-001", "Intra-sentential",
        "Large portion dan sedap.",
        "English phrase + Malay evaluation",
        "Evaluation",
        "The English phrase 'Large portion' is followed by Malay "
        "'dan sedap', evaluating quantity and taste within one sentence."
    ),
    (
        "E02", "REV-005", "Intra-sentential",
        "Service bagus",
        "English phrase + Malay evaluation",
        "Evaluation",
        "The English noun 'Service' is followed by Malay 'bagus', "
        "giving a positive service evaluation."
    ),
    (
        "E03", "REV-008", "Intra-sentential",
        "First time try and tak menyesal sebab ayam gepuk sedap and service bagus.",
        "Alternating English and Malay clauses or phrases",
        "Evaluation",
        "English 'First time try and' introduces a Malay evaluation "
        "'tak menyesal sebab ...', followed by further mixed wording. "
        "The evidence goes beyond the food name."
    ),
    (
        "E04", "REV-018", "Intra-sentential",
        "Confirm repeat lagi!",
        "English return expression + Malay adverb",
        "Intention to return",
        "English 'Confirm repeat' combines with Malay 'lagi' to "
        "express an intention to return or repeat the experience."
    ),
    (
        "E05", "REV-022", "Intra-sentential",
        "Food ok , tapi time saya datang service agak lambat.",
        "Mixed evaluation linked by tapi",
        "Contrast",
        "The passage contrasts an acceptable food assessment with "
        "slow service. Malay 'tapi' links the contrast, with English "
        "'time' and 'service' embedded in the following wording."
    ),
    (
        "E06", "REV-175", "Intra-sentential",
        "Please lah tolong check betul2 before hntr kt driver grab.",
        "Alternating English and Malay clauses or phrases",
        "Request",
        "English 'Please', 'check', and 'before' occur with Malay "
        "'lah tolong', 'betul2', and abbreviated 'hntr kt'. "
        "The reviewer requests more careful order checking."
    ),
    (
        "E07", "REV-006", "Inter-sentential",
        None,
        "Malay evaluation followed by English return expression",
        "Evaluation and intention to return",
        "Under the team's punctuation rule, the Malay evaluation "
        "ending 'memang sedap ah' is separated from English "
        "'confirm repeat' by exclamation punctuation and an emoji."
    ),
    (
        "E08", "REV-016", "Inter-sentential",
        None,
        "Language alternation across line breaks",
        "Evaluation",
        "The Malay line 'Ayam gepuk yang lazat' is followed by "
        "'Services tip top' on a new line, then 'Sis puas hati'. "
        "This interpretation follows the existing annotation and "
        "the team's line-break convention."
    ),
    (
        "E09", "REV-054", "Inter-sentential",
        None,
        "Malay evaluation followed by English recommendation",
        "Evaluation and recommendation",
        "Malay 'sedap' and English 'recommend' are separated by "
        "exclamation punctuation. This counts as inter-sentential "
        "under the team's convention allowing short evaluative fragments."
    ),
    (
        "E10", "REV-130", "Inter-sentential",
        None,
        "Malay evaluation followed by English recommendation",
        "Evaluation and recommendation",
        "Malay 'SEDAP GILA!!!' is followed by English 'Must try ok?'. "
        "The recommendation follows the taste evaluation across "
        "a sentence boundary."
    ),
    (
        "E11", "REV-132", "Inter-sentential",
        None,
        "English evaluation followed by Malay evaluation",
        "Reiteration of evaluation",
        "English 'Taste very delicious.' is followed by Malay "
        "'Sangat sedap'. Both segments give a similar positive "
        "taste evaluation across a sentence boundary."
    ),
]

evidence_rows = []

for (
    evidence_id, review_id, observed_type, quote,
    pattern_group, function, interpretation
) in evidence_specs:

    if review_id not in source.index:
        raise ValueError(f"Missing source review: {review_id}")

    original = source.loc[review_id]

    # Use the complete review for the five inter-sentential cases.
    if quote is None:
        quote = original["Review Text"]

    if quote not in original["Review Text"]:
        raise ValueError(
            f"Draft quote does not exactly match {review_id}. "
            "Check the current dataset before proceeding."
        )

    evidence_rows.append({
        "evidence_id": evidence_id,
        "review_id": review_id,
        "source_row": original["source_row"],
        "evidence_quote": quote,
        "quote_occurrence": 1,
        "primary_switching_type": original["Code-Switching Type"],
        "observed_switch_type": observed_type,
        "pattern_group": pattern_group,
        "contextual_function": function,
        "draft_interpretation": interpretation,
        "sentiment": original["Sentiment"],
        "full_review": original["Review Text"],
        "existing_notes": original["Notes / Annotator Comments"],
        "review_decision": "",
        "reviewer_notes": "",
        "select_for_report": "",
    })

evidence_draft = pd.DataFrame(evidence_rows)

display(
    evidence_draft[
        ["evidence_id", "review_id", "evidence_quote",
         "draft_interpretation"]
    ]
)

### Prepare small lexicon

In [26]:
lexicon_draft = pd.DataFrame([
    (
        "L01", "E01", "dan sedap", "Malay",
        "and tasty/delicious",
        "Adds a taste evaluation after an English phrase."
    ),
    (
        "L02", "E02", "Service bagus", "English–Malay",
        "good service",
        "Evaluates service using an English noun and Malay descriptor."
    ),
    (
        "L03", "E03", "tak menyesal", "Malay",
        "do not regret it",
        "Expresses satisfaction within a mixed-language review passage."
    ),
    (
        "L04", "E04", "Confirm repeat lagi", "English–Malay",
        "will definitely repeat or return again",
        "Expresses an intention to return."
    ),
    (
        "L05", "E05", "service agak lambat", "English–Malay",
        "service was rather slow",
        "Expresses a negative service assessment."
    ),
    (
        "L06", "E06", "check betul2", "English–Malay",
        "check properly or carefully",
        "Combines an English verb with an informal Malay expression; "
        "'betul2' represents 'betul-betul'."
    ),
], columns=[
    "lexicon_id",
    "evidence_id",
    "expression",
    "language_composition",
    "meaning_in_context",
    "usage_note",
])

lexicon_draft["review_decision"] = ""
lexicon_draft["reviewer_notes"] = ""

display(lexicon_draft)

### Save without overwriting review

In [27]:
if SELECTED_REVIEW_FILE.exists():
    print("Existing review workbook preserved:")
else:
    with pd.ExcelWriter(
        SELECTED_REVIEW_FILE,
        engine="openpyxl",
    ) as writer:
        evidence_draft.to_excel(
            writer, sheet_name="Evidence", index=False
        )
        lexicon_draft.to_excel(
            writer, sheet_name="Lexicon", index=False
        )

    print("Created prefilled review workbook:")

print(SELECTED_REVIEW_FILE.resolve())

Existing review workbook preserved:
D:\PROJECT\CPC353 Assignment 1\CPC-354-Assignment-1\Outputs\Tables\selected_evidence_review.xlsx


## Read and validate the review workbook

In [28]:
evidence_review = pd.read_excel(
    SELECTED_REVIEW_FILE,
    sheet_name="Evidence",
    keep_default_na=False,
)

lexicon_review = pd.read_excel(
    SELECTED_REVIEW_FILE,
    sheet_name="Lexicon",
    keep_default_na=False,
)

required_evidence = set(evidence_draft.columns)
required_lexicon = set(lexicon_draft.columns)

if not required_evidence.issubset(evidence_review.columns):
    raise ValueError("Evidence tab is missing required columns.")

if not required_lexicon.issubset(lexicon_review.columns):
    raise ValueError("Lexicon tab is missing required columns.")

# Check that rows have not accidentally been lost or duplicated.
if (
    evidence_review["evidence_id"].duplicated().any()
    or set(evidence_review["evidence_id"])
       != set(evidence_draft["evidence_id"])
):
    raise ValueError("Evidence IDs do not match the prepared review form.")

if (
    lexicon_review["lexicon_id"].duplicated().any()
    or set(lexicon_review["lexicon_id"])
       != set(lexicon_draft["lexicon_id"])
):
    raise ValueError("Lexicon IDs do not match the prepared review form.")

for form in [evidence_review, lexicon_review]:
    form["review_decision"] = (
        form["review_decision"].astype(str).str.strip().str.title()
    )

    if not form["review_decision"].isin(
        ["", "Accept", "Reject"]
    ).all():
        raise ValueError("Use Accept, Reject, or blank for decisions.")

print("Evidence review:")
display(evidence_review["review_decision"].replace(
    "", "Pending"
).value_counts())

print("Lexicon review:")
display(lexicon_review["review_decision"].replace(
    "", "Pending"
).value_counts())

if (
    evidence_review["review_decision"].eq("").any()
    or lexicon_review["review_decision"].eq("").any()
):
    raise ValueError(
        "Some decisions are pending. Review the remaining rows, "
        "save the workbook, and rerun this cell."
    )

accepted = evidence_review.loc[
    evidence_review["review_decision"].eq("Accept")
].copy()

if accepted.empty:
    raise ValueError("No evidence has been accepted.")

expected_ids = evidence_draft.set_index("evidence_id")["review_id"]
allowed_types = {
    "Intra-sentential", "Inter-sentential", "Both", "Unclear"
}

validated_rows = []

for _, row in accepted.iterrows():
    evidence_id = row["evidence_id"]
    review_id = row["review_id"]

    if review_id != expected_ids.loc[evidence_id]:
        raise ValueError(f"Source review ID changed for {evidence_id}.")

    original = source.loc[review_id]
    text = original["Review Text"]
    quote = str(row["evidence_quote"])

    if not quote.strip():
        raise ValueError(f"Empty quotation: {evidence_id}")

    matches = list(re.finditer(re.escape(quote), text))

    try:
        number = float(str(row["quote_occurrence"]).strip() or "1")
        if not number.is_integer() or number < 1:
            raise ValueError
        occurrence = int(number)
    except (ValueError, OverflowError):
        raise ValueError(f"Invalid quote occurrence: {evidence_id}")

    if occurrence > len(matches):
        raise ValueError(
            f"Exact quotation not found for {evidence_id}."
        )

    for field in [
        "pattern_group",
        "contextual_function",
        "draft_interpretation",
    ]:
        if not str(row[field]).strip():
            raise ValueError(f"Missing {field}: {evidence_id}")

    if row["observed_switch_type"] not in allowed_types:
        raise ValueError(f"Invalid switching type: {evidence_id}")

    selected = str(row["select_for_report"]).strip().lower()
    if selected not in {"", "yes", "no"}:
        raise ValueError("Use Yes, No, or blank for select_for_report.")

    match = matches[occurrence - 1]

    validated_rows.append({
        "evidence_id": evidence_id,
        "review_id": review_id,
        "source_row": original["source_row"],
        "passage_id": f"{review_id}:{match.start()}:{match.end()}",
        "quote": quote,
        "primary_switching_type": original["Code-Switching Type"],
        "observed_switch_type": row["observed_switch_type"],
        "pattern_group": row["pattern_group"],
        "function": row["contextual_function"],
        "interpretation": row["draft_interpretation"],
        "sentiment": original["Sentiment"],
        "reviewer_notes": row["reviewer_notes"],
        "selected_for_report": selected == "yes",
    })

selected_evidence = pd.DataFrame(validated_rows)

if selected_evidence["passage_id"].duplicated().any():
    raise ValueError("Duplicate selected passages need review.")

display(selected_evidence)

Evidence review:
Lexicon review:


## Summarize the selected patterns

In [29]:
selected_pattern_summary = (
    selected_evidence.groupby("pattern_group")
    .agg(
        selected_passages=("passage_id", "nunique"),
        selected_reviews=("review_id", "nunique"),
        example_review_ids=(
            "review_id",
            lambda values: ", ".join(sorted(set(values))),
        ),
    )
    .reset_index()
    .sort_values("selected_reviews", ascending=False)
)

selected_pattern_summary["scope"] = (
    "Purposively selected evidence; not corpus-wide pattern frequency"
)

display(selected_pattern_summary)

selected_evidence.to_csv(
    TABLE_DIR / "selected_evidence_analysis.csv",
    index=False,
    encoding="utf-8-sig",
)

selected_pattern_summary.to_csv(
    TABLE_DIR / "selected_pattern_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

## Produce the inter-sentential analysis

In [30]:
inter_case_ids = {"E07", "E08", "E09", "E10", "E11"}

inter_case_review = evidence_review.loc[
    evidence_review["evidence_id"].isin(inter_case_ids),
    [
        "evidence_id",
        "review_id",
        "review_decision",
        "reviewer_notes",
    ],
]

display(inter_case_review)

inter_analysis = selected_evidence.loc[
    selected_evidence["evidence_id"].isin(inter_case_ids),
    [
        "review_id",
        "quote",
        "primary_switching_type",
        "observed_switch_type",
        "function",
        "interpretation",
        "sentiment",
    ],
].copy()

inter_analysis.to_csv(
    TABLE_DIR / "inter_sentential_analysis.csv",
    index=False,
    encoding="utf-8-sig",
)

display(inter_analysis)

rejected_inter = inter_case_review.loc[
    inter_case_review["review_decision"].eq("Reject")
]

if not rejected_inter.empty:
    print(
        "Some draft interpretations were rejected. Their original "
        "dataset labels remain unchanged. Explain these cases separately."
    )

## Export the reviewed bilingual lexicon

In [31]:
accepted_lexicon = lexicon_review.loc[
    lexicon_review["review_decision"].eq("Accept")
].copy()

evidence_lookup = selected_evidence.set_index("evidence_id")
lexicon_rows = []

for _, row in accepted_lexicon.iterrows():
    evidence_id = row["evidence_id"]

    if evidence_id not in evidence_lookup.index:
        raise ValueError(
            f"{row['lexicon_id']} cites evidence that was not accepted."
        )

    evidence = evidence_lookup.loc[evidence_id]
    expression = str(row["expression"])

    if not expression.strip() or expression not in evidence["quote"]:
        raise ValueError(
            f"Expression is not an exact part of its evidence: "
            f"{row['lexicon_id']}"
        )

    for field in [
        "language_composition",
        "meaning_in_context",
        "usage_note",
    ]:
        if not str(row[field]).strip():
            raise ValueError(
                f"Missing {field} in {row['lexicon_id']}"
            )

    lexicon_rows.append({
        "expression": expression,
        "language_composition": row["language_composition"],
        "meaning_in_context": row["meaning_in_context"],
        "usage_note": row["usage_note"],
        "example_review_id": evidence["review_id"],
        "example_quote": evidence["quote"],
        "reviewer_notes": row["reviewer_notes"],
    })

lexicon = pd.DataFrame(lexicon_rows, columns=[
    "expression",
    "language_composition",
    "meaning_in_context",
    "usage_note",
    "example_review_id",
    "example_quote",
    "reviewer_notes",
])

lexicon.to_csv(
    TABLE_DIR / "bilingual_lexicon.csv",
    index=False,
    encoding="utf-8-sig",
)

display(lexicon)

## Export quotations, distributions, and figure

In [32]:
# Export final table result
report_quotes = selected_evidence.loc[
    selected_evidence["selected_for_report"],
    [
        "review_id",
        "quote",
        "pattern_group",
        "observed_switch_type",
        "function",
        "interpretation",
    ],
].copy()

report_quotes.to_csv(
    TABLE_DIR / "selected_quotations.csv",
    index=False,
    encoding="utf-8-sig",
)

language_summary.to_csv(
    TABLE_DIR / "language_distribution.csv",
    index=False,
)

switching_summary.to_csv(
    TABLE_DIR / "primary_switching_distribution.csv",
    index=False,
)

keyword_summary.to_csv(
    TABLE_DIR / "keyword_search_summary.csv",
    index=False,
)

# Export final figure
fig, ax = plt.subplots(figsize=(7, 4))

ax.barh(
    switching_summary["primary_switching_type"],
    switching_summary["review_count"],
)

ax.set_xlabel("Number of reviews")
ax.set_title("Primary switching labels in code-switched reviews")

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "primary_switching_distribution.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()
display(report_quotes)

## 4. Sentiment distributions

## Table 1: overall sentiment

Percentage = count / total reviews × 100. Rounding may make displayed percentages sum to 100.01%.


In [33]:
overall = df["Sentiment"].value_counts().reindex(SENTIMENTS, fill_value=0).rename("review_count").to_frame()
overall.index.name = "sentiment"
overall["percentage"] = (overall["review_count"] / len(df) * 100).round(2)
overall["denominator"] = len(df)
display(overall)


## Table 2: sentiment by primary switching type

Each percentage uses that switching group's total, not all reviews. N/A is the non-switching comparison group. Unobserved switching categories are omitted; absent sentiments within an observed group remain zero. Reviews containing both switching types retain their assigned primary label.


In [34]:
observed_types = [t for t in SWITCH_TYPES if t in set(df["Code-Switching Type"])]
counts = pd.crosstab(df["Code-Switching Type"], df["Sentiment"]).reindex(index=observed_types, columns=SENTIMENTS, fill_value=0)
counts.index.name = "switching_type"
totals = counts.sum(axis=1)
percentages = counts.div(totals, axis=0).mul(100).round(2)
report = pd.DataFrame(index=counts.index)
for sentiment in SENTIMENTS:
    report[sentiment + " n (%)"] = [f"{n} ({p:.2f}%)" for n, p in zip(counts[sentiment], percentages[sentiment])]
report["Total"] = totals
display(report)
assert counts.to_numpy().sum() == len(df)
assert counts.sum(axis=0).equals(overall["review_count"].rename_axis("Sentiment"))
assert (percentages.sum(axis=1) - 100).abs().lt(0.02).all()


## Export reproducible tables

Numeric counts and percentages are exported separately for reuse, alongside the formatted report table. Only this notebook's output files are replaced on rerun.


In [35]:
overall.to_csv(TABLE_DIR / "sentiment_distribution.csv", encoding="utf-8-sig")
counts.assign(Total=totals).to_csv(TABLE_DIR / "sentiment_by_switching_counts.csv", encoding="utf-8-sig")
percentages.assign(denominator=totals).to_csv(TABLE_DIR / "sentiment_by_switching_percentages.csv", encoding="utf-8-sig")
report.to_csv(TABLE_DIR / "sentiment_by_switching_report.csv", encoding="utf-8-sig")
print("Exported four sentiment tables.")


Exported four sentiment tables.


## Figures

The first chart shows review counts. The second compares proportions and displays group sizes; the inter-sentential group is very small, so its percentage must be interpreted cautiously.


In [36]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(SENTIMENTS, overall["review_count"], color=COLORS)
for bar, n, p in zip(bars, overall["review_count"], overall["percentage"]):
    ax.text(bar.get_x() + bar.get_width()/2, n + len(df)*0.015, f"{n} ({p:.2f}%)", ha="center")
ax.set_ylim(0, max(overall["review_count"])*1.22)
ax.set_ylabel("Number of reviews")
ax.set_title(f"Overall sentiment (n = {len(df)})")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sentiment_distribution.png", dpi=300)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
left = pd.Series(0.0, index=counts.index)
for sentiment, color in zip(SENTIMENTS, COLORS):
    widths = counts[sentiment] / totals * 100
    ax.barh(counts.index, widths, left=left, label=sentiment, color=color)
    for i, (start, width) in enumerate(zip(left, widths)):
        if width >= 8:
            ax.text(start + width/2, i, f"{width:.1f}%", ha="center", va="center", color="white")
    left += widths
ax.set_yticks(range(len(totals)), [f"{t} (n={n})" for t, n in totals.items()])
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Percentage within switching group")
ax.set_title("Sentiment by primary switching type")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=3, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sentiment_by_switching.png", dpi=300)
plt.show()
plt.close(fig)


## Interpretation and handoff

Use the computed tables to describe the sample, then select passages from the concordance output to explain negation, praise, complaints, contrast and expectations. A keyword occurrence is not a sentiment label or automatically a switching event. Passage sentiment may differ from the overall review sentiment.

In the initial 197-review snapshot, intra-sentential and non-switching reviews have similar positive shares. All five inter-sentential reviews are positive, but the small group does not justify a broad claim about that switching type. These observations do not establish causation.

Phing Hong's switching-distribution percentages use only code-switched reviews as their denominator; this notebook's within-group percentages answer a different question.


## 5. Recurring expressions in context
Six expressions are purposively selected from recurring N-grams for praise, complaints, return intentions and contrast. All occurrences of these six are inspected; this is not a random sample or a census of all bilingual constructions. New interpretations are explicitly pending team review.

In [37]:
# Purposive selection of recurring evaluative and contrast expressions.
# Decisions below are assistant interpretations for team review, not independent annotation.
selected_terms = ['service bagus', 'service teruk', 'repeat lagi', 'portion puas hati', 'tak puas hati', 'sedap tapi']
ng_hits = pd.read_csv(TABLE_DIR / 'ngram_occurrences.csv', keep_default_na=False)
ng_freq = pd.read_csv(TABLE_DIR / 'ngram_frequencies.csv', keep_default_na=False)
context_hits = ng_hits.loc[ng_hits['ngram'].isin(selected_terms)].copy()
assert set(context_hits['ngram']) == set(selected_terms)
context_source = df.set_index('Review ID')
interpretations = {
 'service bagus': ('Mixed within phrase', 'service / bagus', 'Service', 'Positive', 'English service combines with Malay bagus to praise service.'),
 'service teruk': ('Mixed within phrase', 'service / teruk', 'Service', 'Negative', 'English service combines with Malay teruk to criticise service.'),
 'repeat lagi': ('Mixed within phrase', 'repeat / lagi', 'Return intention', 'Positive', 'English repeat combines with Malay lagi; the surrounding passages express intention to return.'),
 'portion puas hati': ('Mixed within phrase', 'portion / puas hati', 'Portion size/value', 'Positive', 'English portion combines with Malay satisfaction wording; the context praises portion satisfaction.'),
 'tak puas hati': ('Malay-only phrase; inspect wider context', 'tak puas hati', 'Complaint', 'Negative', 'Malay negation tak makes the satisfaction expression negative. Switching evidence comes from surrounding English wording, not these three words alone.'),
 'sedap tapi': ('Malay-only phrase; inspect wider context', 'sedap tapi', 'Contrast', 'Context dependent', 'Both words are Malay. Tapi introduces a qualification or contrast; its outcome must be read in context.'),
}
specific = {
 ('sedap tapi','REV-021'): ('Food/service','Positive elaboration','The review contrasts usual good food with an unusually positive service recovery; tapi does not introduce a negative overall judgement here.'),
 ('sedap tapi','REV-025'): ('Food consistency','Positive elaboration','Bukan sekadar sedap, tapi consistent, reliable adds praise; English descriptors in the continuation provide switching evidence.'),
 ('sedap tapi','REV-034'): ('Missing sambal','Mixed passage / negative outcome','Praise of sambal contrasts with its omission from the order and refusal to buy again; craving and take away occur in the wider mixed segment.'),
 ('sedap tapi','REV-089'): ('Chicken taste/texture','Mixed passage / negative outcome','Initial praise is qualified by bland chicken; English crispy occurs later in the same segment.'),
 ('sedap tapi','REV-091'): ('Food/communication','Mixed','The segment contrasts tasty food with communication needing improvement, but is Malay-only. Cashier tidak peka in the next sentence establishes switching elsewhere.'),
}
context_rows=[]
for r in context_hits.itertuples():
 original=context_source.loc[r.review_id]
 text=original['Review Text']
 assert text[r.match_start:r.match_end] == r.source_quote
 assert original['Sentiment']==r.sentiment
 status,spans,target,polarity,note=interpretations[r.ngram]
 if (r.ngram,r.review_id) in specific:target,polarity,note=specific[(r.ngram,r.review_id)]
 if r.ngram=='tak puas hati':
  target={'REV-112':'Outside-drink policy','REV-160':'Counter staff','REV-161':'Food/order experience'}[r.review_id]
  note += ' Overall Neutral in REV-112 reflects mixed evaluations; it does not make this complaint neutral.' if r.review_id=='REV-112' else ''
 context_rows.append(dict(occurrence_id=f'{r.review_id}:{r.n}:{r.match_start}:{r.match_end}',review_id=r.review_id,ngram=r.ngram,n=r.n,source_quote=r.source_quote,segment_text=r.segment_text,full_review=text,primary_switching_type=original['Code-Switching Type'],overall_sentiment=original['Sentiment'],phrase_language_assessment=status,language_spans=spans,evaluation_target=target,passage_evaluation=polarity,interpretation=note,review_status='Assistant interpretation - pending team review'))
context_evidence=pd.DataFrame(context_rows)
assert context_evidence.occurrence_id.is_unique
context_summary=context_evidence.groupby(['n','ngram']).agg(occurrences=('occurrence_id','size'),distinct_reviews=('review_id','nunique')).reset_index()
check=context_summary.merge(ng_freq,on=['n','ngram'],suffixes=('_context','_all'),validate='one_to_one')
assert check.occurrences_context.eq(check.occurrences_all).all()
assert check.distinct_reviews_context.eq(check.distinct_reviews_all).all()
for sentiment in ['Positive','Negative','Neutral']:
 ids=context_evidence.loc[context_evidence.overall_sentiment.eq(sentiment)].groupby('ngram').review_id.nunique()
 context_summary[sentiment+'_reviews']=context_summary.ngram.map(ids).fillna(0).astype(int)
assert context_summary[['Positive_reviews','Negative_reviews','Neutral_reviews']].sum(axis=1).eq(context_summary.distinct_reviews).all()
context_evidence.to_csv(TABLE_DIR/'phase3_contextual_evidence.csv',index=False,encoding='utf-8-sig')
context_summary.to_csv(TABLE_DIR/'phase3_expression_summary.csv',index=False,encoding='utf-8-sig')
# Validate the previously prepared 12 passages without changing their interpretations.
integrated=pd.read_csv(TABLE_DIR/'integrated_discussion_evidence.csv',keep_default_na=False)
for r in integrated.itertuples():
 current=context_source.loc[r.review_id]
 assert r.quote in current['Review Text']
 assert r.overall_sentiment==current['Sentiment']
 assert r.primary_switching_type==current['Code-Switching Type']
print(context_summary.to_string(index=False))
print('Validated',len(context_evidence),'occurrences and',len(integrated),'existing discussion passages.')


 n             ngram  occurrences  distinct_reviews  Positive_reviews  Negative_reviews  Neutral_reviews
 2       repeat lagi            3                 3                 3                 0                0
 2        sedap tapi            5                 5                 2                 2                1
 2     service bagus            5                 5                 5                 0                0
 2     service teruk            3                 2                 0                 2                0
 3 portion puas hati            2                 2                 2                 0                0
 3     tak puas hati            4                 3                 0                 2                1
Validated 22 occurrences and 12 existing discussion passages.


In [38]:
for path, expected in protected_hashes.items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected, f'Protected input changed: {path}'
print('Source workbook and manual evidence inputs unchanged.')

Source workbook and manual evidence inputs unchanged.
